# Topic 19: Convolutional Neural Networks: Filters, Feature Maps, Sizes, and Pooling

CSCI 3320 · AI Programming in Python · Fall 2026

This notebook is the code from the [Topic 19 lecture note](https://dongchul.kim/csci3320-fall2026/topic/21), every cell in the
same order, under the note's own section headings. The explanations are in the note: keep it
open beside this notebook. Run the cells from top to bottom (Runtime > Run all works too),
then change things and run them again.

It runs on a CPU; no GPU is needed. Numbers can differ a little from the note on a
different library version.

## One filter on one patch

In [ ]:
import numpy as np

patch = np.array([[0, 0, 1],
                  [0, 0, 1],
                  [0, 0, 1]])
filt = np.array([[-1, 0, 1],
                 [-1, 0, 1],
                 [-1, 0, 1]])

products = patch * filt        # multiply position by position
print(products)
print("sum:", int(products.sum()))

## The filter slides over the image

In [ ]:
import numpy as np

def convolve(image, filt):
    h, w = image.shape
    k = filt.shape[0]                            # the filter is k x k
    out = np.zeros((h - k + 1, w - k + 1), dtype=int)
    for r in range(h - k + 1):                   # every row the filter fits in
        for c in range(w - k + 1):               # every column it fits in
            patch = image[r:r + k, c:c + k]      # the k x k patch under the filter
            out[r, c] = (patch * filt).sum()     # multiply and add: one number
    return out

image = np.array([[0, 0, 0, 1, 1, 1]] * 6)       # 6 x 6: dark, then bright
filt = np.array([[-1, 0, 1]] * 3)                # 3 x 3: a vertical edge
print(convolve(image, filt))

In [ ]:
import numpy as np

def convolve(image, filt):
    h, w = image.shape
    k = filt.shape[0]
    out = np.zeros((h - k + 1, w - k + 1), dtype=int)
    for r in range(h - k + 1):
        for c in range(w - k + 1):
            out[r, c] = (image[r:r + k, c:c + k] * filt).sum()
    return out

seven = np.array([[0, 0, 0, 0, 0, 0],
                  [0, 1, 1, 1, 1, 0],
                  [0, 0, 0, 0, 1, 0],
                  [0, 0, 0, 1, 0, 0],
                  [0, 0, 1, 0, 0, 0],
                  [0, 0, 1, 0, 0, 0]])
vertical = np.array([[-1, 0, 1]] * 3)
horizontal = np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]])

print("vertical edge filter:")
print(convolve(seven, vertical))
print("horizontal edge filter:")
print(convolve(seven, horizontal))

## The same calculation in PyTorch

In [ ]:
import torch
import torch.nn.functional as F

row = [0., 0., 0., 1., 1., 1.]
image = torch.tensor([row] * 6)                  # a 6 x 6 image
filt = torch.tensor([[-1., 0., 1.]] * 3)         # a 3 x 3 filter

x = image.reshape(1, 1, 6, 6)    # batch, channels, height, width
w = filt.reshape(1, 1, 3, 3)     # output channels, input channels, height, width

out = F.conv2d(x, w)
print(out.shape)
print(out[0, 0])

## Sizes: padding, stride, and one formula

### The formula

In [ ]:
import numpy as np

image = np.array([[0, 0, 0, 1, 1, 1]] * 6)
padded = np.pad(image, 1)        # one ring of zeros on every side
print("padded shape:", padded.shape)
print(padded)

def output_size(n, k, p=0, s=1):
    return (n - k + 2 * p) // s + 1              # // rounds down

for n, k, p, s in [(6, 3, 0, 1), (6, 3, 1, 1), (28, 3, 1, 1), (28, 3, 1, 2), (28, 3, 0, 2)]:
    print(f"n={n:2d} k={k} p={p} s={s}  ->  {output_size(n, k, p, s)}")

In [ ]:
import torch
import torch.nn as nn

x = torch.zeros(1, 1, 28, 28)    # a blank image: we only want the shape
for k, p, s in [(3, 0, 1), (3, 1, 1), (5, 2, 1), (3, 1, 2), (3, 0, 2)]:
    conv = nn.Conv2d(1, 1, kernel_size=k, padding=p, stride=s)
    print(k, p, s, tuple(conv(x).shape))

## Many filters give many channels

In [ ]:
import torch
import torch.nn as nn

conv = nn.Conv2d(1, 8, 3, padding=1)
x = torch.zeros(1, 1, 28, 28)    # batch, channels, height, width
print(conv(x).shape)             # 8 maps out
print(conv.weight.shape)         # 8 filters, each 1 x 3 x 3

color = nn.Conv2d(3, 8, 3, padding=1)
y = torch.zeros(1, 3, 32, 32)    # a 32 x 32 color photo
print(color(y).shape)
print(color.weight.shape)        # 8 filters, each 3 x 3 x 3

## Counting the weights

In [ ]:
def conv_weights(k, channels_in, channels_out):
    one_filter = k * k * channels_in + 1         # its weights, plus 1 bias
    return one_filter * channels_out

for k, a, b in [(3, 1, 8), (3, 8, 16), (3, 3, 8), (5, 3, 32)]:
    print(f"Conv2d({a}, {b}, {k}):  {conv_weights(k, a, b)} weights")

In [ ]:
import torch.nn as nn

for layer in [nn.Conv2d(1, 8, 3), nn.Conv2d(8, 16, 3), nn.Conv2d(3, 8, 3), nn.Conv2d(3, 32, 5)]:
    print(layer.in_channels, layer.out_channels, sum(p.numel() for p in layer.parameters()))

## Pooling: shrink the map

In [ ]:
import numpy as np

def max_pool(fmap, size=2):
    h, w = fmap.shape
    out = np.zeros((h // size, w // size), dtype=int)
    for r in range(h // size):
        for c in range(w // size):
            block = fmap[r * size:(r + 1) * size, c * size:(c + 1) * size]
            out[r, c] = block.max()              # keep only the largest
    return out

fmap = np.array([[1, 3, 0, 2],
                 [5, 2, 1, 0],
                 [0, 1, 4, 6],
                 [2, 0, 3, 1]])
print(max_pool(fmap))

a = np.zeros((4, 4), dtype=int); a[0, 0] = 9     # a strong answer here
b = np.zeros((4, 4), dtype=int); b[1, 1] = 9     # the same answer, one pixel away
print(max_pool(a))
print(max_pool(b))

In [ ]:
import torch
import torch.nn as nn

pool = nn.MaxPool2d(2)
fmap = torch.tensor([[1., 3., 0., 2.],
                     [5., 2., 1., 0.],
                     [0., 1., 4., 6.],
                     [2., 0., 3., 1.]])
print(pool(fmap.reshape(1, 1, 4, 4))[0, 0])
print("weights:", sum(p.numel() for p in pool.parameters()))

many = torch.zeros(1, 8, 28, 28)        # 8 channels
print(pool(many).shape)                 # halved; still 8 channels

## The whole network

In [ ]:
import torch
import torch.nn as nn

model = nn.Sequential(
    nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(16 * 7 * 7, 10),
)

x = torch.zeros(1, 1, 28, 28)
for layer in model:
    x = layer(x)
    weights = sum(p.numel() for p in layer.parameters())
    print(f"{type(layer).__name__:10s} {str(tuple(x.shape)):18s} {weights:5d} weights")
print("total:", sum(p.numel() for p in model.parameters()))

## The most common error

In [ ]:
import torch
import torch.nn as nn

bad = nn.Sequential(
    nn.Conv2d(1, 8, 3, padding=1),
    nn.Flatten(),
    nn.Linear(100, 10),                 # wrong: what really arrives is 8 * 28 * 28
)
try:
    bad(torch.zeros(1, 1, 28, 28))
except RuntimeError as e:
    print("RuntimeError:", e)

good = nn.Sequential(
    nn.Conv2d(1, 8, 3, padding=1),
    nn.Flatten(),
    nn.Linear(8 * 28 * 28, 10),
)
print(good(torch.zeros(1, 1, 28, 28)).shape)

Back to the [Topic 19 lecture note](https://dongchul.kim/csci3320-fall2026/topic/21).